# Sentiment Analysis with SpaCy

In [19]:
import pandas as pd
import spacy

from spacytextblob.spacytextblob import SpacyTextBlob

In [20]:
# load medium english model
# this model helps with semantic understanding
nlp = spacy.load("en_core_web_md")

nlp.add_pipe("spacytextblob")

In [21]:
# Load dataset and display first 5 rows
# I limited the data to the first 5000 rows to reduce runtime
amazon_df = pd.read_csv("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv", encoding="latin1", nrows=5000)
amazon_df.head()

,ï»¿id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.didPurchase,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs
0,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,3,https://www.amazon.com/product-reviews/B00QWO9...,I order 3 of them and one of the item is bad q...,... 3 of them and one of the item is bad quali...,Byger yang,"https://www.barcodable.com/upc/841710106442,ht..."
1,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,4,https://www.amazon.com/product-reviews/B00QWO9...,Bulk is always the less expensive way to go fo...,... always the less expensive way to go for pr...,ByMG,"https://www.barcodable.com/upc/841710106442,ht..."
2,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Well they are not Duracell but for the price i...,... are not Duracell but for the price i am ha...,BySharon Lambert,"https://www.barcodable.com/upc/841710106442,ht..."
3,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Seem to work as well as name brand batteries a...,... as well as name brand batteries at a much ...,Bymark sexson,"https://www.barcodable.com/upc/841710106442,ht..."
4,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,These batteries are very long lasting the pric...,... batteries are very long lasting the price ...,Bylinda,"https://www.barcodable.com/upc/841710106442,ht..."


In [22]:
# Extracts the "reviews.text" column
reviews_data = amazon_df['reviews.text']

# Display first few reviews
reviews_data.head()


0    I order 3 of them and one of the item is bad q...
1    Bulk is always the less expensive way to go fo...
2    Well they are not Duracell but for the price i...
3    Seem to work as well as name brand batteries a...
4    These batteries are very long lasting the pric...
Name: reviews.text, dtype: str

In [23]:
# Removes rows with missing data
clean_data = amazon_df.dropna(subset=['reviews.text'])

In [24]:
def preprocess_text(text):
    """
    This function cleans the text by:
    - converting to strings, removing whitespace and changing to lowercase
    - removing all punctuation using spaCy
    """

    # Basic cleaning
    text = str(text).lower().strip()

    # process text with spaCy
    doc = nlp(text)

    # Filter tokens
    cleaned_tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct
    ]

    # Join tokens into cleaned string
    return " ".join(cleaned_tokens)

In [25]:
# Apply predict_text to review text
clean_data["cleaned_reviews"] = clean_data["reviews.text"].apply(preprocess_text)

# print results
clean_data[["reviews.text", "cleaned_reviews"]].head()

,reviews.text,cleaned_reviews
0,I order 3 of them and one of the item is bad q...,order 3 item bad quality miss backup spring pc...
1,Bulk is always the less expensive way to go fo...,bulk expensive way product like
2,Well they are not Duracell but for the price i...,duracell price happy
3,Seem to work as well as name brand batteries a...,work brand battery well price
4,These batteries are very long lasting the pric...,battery long last price great


In [26]:
def analyze_sentiment(review):
    """
    This function processes the review using spaCy.
    Uses polarity score to classify sentiment
    """

    # Cleans input text
    review = str(review).lower().strip()

    # Processes text
    doc = nlp(review)

    # Extract polarity score
    polarity = doc._.blob.polarity

    # Using the sentiment attribute
    # sentiment = doc._.blob.sentiment

    if polarity > 0:
        sentiment = "Positive"
    elif polarity < 0:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    # added sentiment to the return statement
    return sentiment, float(polarity)


**Changes made below:**

Split the returned sentiment label and polarity score into two separate columns in the dataset.
Applied the sentiment analysis function to the cleaned review text.

In [27]:
# Apply sentiment function to dataset
clean_data["sentiment"], clean_data["polarity"] = zip(*clean_data["cleaned_reviews"].apply(analyze_sentiment))

clean_data[["reviews.text","cleaned_reviews", "sentiment", "polarity"]].head()

,reviews.text,cleaned_reviews,sentiment,polarity
0,I order 3 of them and one of the item is bad q...,order 3 item bad quality miss backup spring pc...,Negative,-0.70
1,Bulk is always the less expensive way to go fo...,bulk expensive way product like,Negative,-0.50
2,Well they are not Duracell but for the price i...,duracell price happy,Positive,0.80
3,Seem to work as well as name brand batteries a...,work brand battery well price,Neutral,0.00
4,These batteries are very long lasting the pric...,battery long last price great,Positive,0.25


### Similarity Comparison

In [28]:
review1 = nlp("This product is amazing and works perfectly.")
review2 = nlp("I love this item, it works really well.")

similarity_score = review1.similarity(review2)

print("\nSimilarity Comparison:")
print(f"Similarity Score: {similarity_score:.4f}")


Similarity Comparison:
Similarity Score: 0.8750


A score closer to 1, means that the reviews are very similar in meaning, while a score closer to 0 means they are different. 

As we can see above the similarity score is closer to 1, indicating that both reviews are similar in meaning.

### Test on sample reviews

In [29]:
# Test the model on custom reviews
sample_reviews = [
    "This product is amazing!",
    "The product looks nothing like the advertisement, very disappointed.",
    "Average quality"
]

# Display predictions
for review in sample_reviews:
    sentiment, polarity = analyze_sentiment(review)
    print("Review:", review)
    print(f"Predicted Sentiment: {sentiment}")
    print(f"Polarity Score: {polarity:.3f}")
    print("-" * 80)

Review: This product is amazing!
Predicted Sentiment: Positive
Polarity Score: 0.750
--------------------------------------------------------------------------------
Review: The product looks nothing like the advertisement, very disappointed.
Predicted Sentiment: Negative
Polarity Score: -0.975
--------------------------------------------------------------------------------
Review: Average quality
Predicted Sentiment: Negative
Polarity Score: -0.150
--------------------------------------------------------------------------------
